In [17]:
!pip install akshare --upgrade

In [18]:
import akshare as ak
import numpy as np
import pandas as pd

In [19]:
def crr_vanilla_call(S, K, sigma, r, T, n, b=0.0, verbose=False):
  '''
  args:
  S: 当前股票价格，K=执行价格，sigma=年化波动率，r=年化利率，T=总时间（年），n=步数，b=年化股息率
  return:
  当前价值，float
  '''
  dt = T / n
  u = np.exp(sigma * np.sqrt(dt))
  d = 1 / u
  p = (np.exp( (r - b) * dt ) - d) / (u - d)

  i = np.arange(n+1)
  S_terminal = S * (u**i) * (d ** (n-i)) # 终点股票价格数组
  V = np.maximum(S_terminal-K, 0) # 终期价值数组,到期时二选一,取大的

  # 把相邻元素配对计算，使 V 的长度减少1，并打印长度观察变化。
  for j in range(n-1, -1, -1):
    V = np.exp(- r * dt) *(p * V[1:] + (1-p) * V[:-1]) # V = 折现因子 * 加权数组，每轮长度减少1的当前价值数组
    if verbose:
      print(f"当前层:{j}, V的长度: {len(V)}")

  return float(V[0])
# 调试调用
print(crr_vanilla_call(S=100,K=100,sigma=0.2,r=0.05,T=1,n=3, verbose=True))

n_values = [10, 50, 200, 1000]
# 其他参数
params = {
    "S" : 100,
    "K" : 100,
    "sigma" : 0.2,
    "r" : 0.05,
    "T": 1}

'''
price = np.zeros(len(n_values))
# 比较不同步数的计算结果
for k, current_n in enumerate(n_values):
  price[k] = crr_vanilla_call(**params, n=current_n)
  print(f"当前步数:{current_n}, 价格：{price[k]:.6f}")

changes = np.abs(price[1:] - price[:-1])
for k, change in enumerate(changes):
  print(f"n{n_values[k]} -> {n_values[k+1]}, 变化量{change:.6f}")
'''

def compare_crr_steps(n_values, params):
  """
  args:
  n_values=不同步数，params = 其他不变的参数
  return：
  每个步数对应的计算结果，相邻结果的绝对变化量
  """
  prices = np.zeros(len(n_values))
  for k, current_n in enumerate(n_values):
    prices[k] = crr_vanilla_call(**params, n=current_n)

  changes = np.abs(prices[1:] - prices[:-1])
  return prices, changes

prices, changes = compare_crr_steps(n_values, params)
print(prices, changes)




当前层:2, V的长度: 3
当前层:1, V的长度: 2
当前层:0, V的长度: 1
11.043871091951113
[10.25340904 10.41069154 10.44059126 10.4485841 ] [0.1572825  0.02989972 0.00799284]


In [20]:
# 其他参数
params_con = {
    "S" : 100,
    "K" : 100,
    "sigma" : 0.2,
    "r" : 0.05,
    "T" : 1,
    "b" : 0.0,
    "P_R" : 108.0,
    "coupon" : 0.01,
    "spread" : 0.02, # 额外折现率 信用利差？
    "lo" : 70.0, # 插值下界
    "hi" : 130.0, # 插值上界
    "t_conv": 0.5, # 开始允许转换的时间
    "t_put": 0.5 # 允许回售的时间
    }

def parity(S, face, K):
  return S * face / K

def crr_convertible_basic(S, K, sigma, r, T,
                          b, P_R, coupon, spread, lo, hi, t_conv,t_put,
                          n, C1=130, C2=70, face=100, P_put=103.0, verbose=False):
  '''
  args:
  S: 当前股票价格，K=执行价格，sigma=年化波动率，r=年化利率，T=总时间（年），n=步数，b=年化股息率
  face=转债面值, P_R=到期赎回金额, coupon = 年化票面利率, spread = 额外折现率 信用利差？,
  lo = 插值下界, hi = 插值上界, C1 = 强赎触发线（平价），C2 = 回售触发线（平价），
  P_put=回售价格
  return:
  当前价值，float
  '''
  dt = T / n
  u = np.exp(sigma * np.sqrt(dt))
  d = 1 / u
  p = (np.exp( (r - b) * dt ) - d) / (u - d)
  interest_per_step = coupon * face * dt

  i_current = np.arange(n+1)
  S_terminal = S * (u**i_current) * (d ** (n-i_current)) # 终点股票价格数组
  parity_terminal =  parity(S_terminal, face, K)
  V = np.maximum(parity_terminal, P_R) # 终期价值数组,到期时二选一,取大的

  for j in range(n-1, -1, -1): # 从终期价值回溯
    i_current = np.arange(j+1)
    S_current = S * (u ** i_current) * (d ** (j-i_current))
    parity_current = parity(S_current, face, K)
    t_current = j * dt
    # 折现率
    P_ct = np.clip(((parity_current - lo) / (hi - lo)), 0, 1)
    Df = r * P_ct + (1-P_ct) * (r + spread)
    V_EU = np.exp(- Df * dt) *(p * V[1:] + (1-p) * V[:-1]) + interest_per_step

    if t_current >= t_conv: # 时间到了才能换
      hit_call = parity_current > C1 # 检测强赎条款
      V = np.maximum(V_EU, parity_current)
      V[hit_call] = parity_current[hit_call]
    else:
      V = V_EU

    if t_current >= t_put:
      put = parity_current < C2 # 检测回售条款
      V[put] = np.maximum(V_EU[put], P_put)

    if verbose:
      print(f"当前层:{j}, V的长度: {len(V)}")

  return float(V[0])


print(crr_convertible_basic(**params_con, n=5, verbose = True))

当前层:4, V的长度: 5
当前层:3, V的长度: 4
当前层:2, V的长度: 3
当前层:1, V的长度: 2
当前层:0, V的长度: 1
109.17928591664443


In [21]:
bond_zh_cov_df = ak.bond_zh_cov()
print(bond_zh_cov_df)

 67%|██████▋   | 2/3 [00:06<00:03,  3.39s/it]/usr/local/lib/python3.12/dist-packages/akshare/bond/bond_zh_cov.py:342: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  big_df = pd.concat(objs=[big_df, temp_df], ignore_index=True)
                                             

        债券代码   债券简称        申购日期    申购代码  申购上限    正股代码  正股简称    正股价     转股价  \
0     123276  久吾转02  2026-07-20  370631   100  300631  久吾高科  19.45   23.78   
1     118073   赛斯转债  2026-07-17  718480   100  688480   赛恩斯  69.00   88.16   
2     113708  曙26转债  2026-07-15  754019   100  603019  中科曙光  92.46  108.89   
3     110102   江农转债  2026-07-14  733389   100  600389  江山股份  17.77   20.64   
4     123275   肇民转债  2026-07-09  371000   100  301000  肇民科技  25.36   33.80   
...      ...    ...         ...     ...   ...     ...   ...    ...     ...   
1030  110227   赤化转债  2007-10-10  733227   100  600227   赤天化   3.25     NaN   
1031  126006  07深高债  2007-10-09  733548   100  600548   深高速   8.58     NaN   
1032  110971   恒源转债  2007-09-24  733971   100  600971  恒源煤电   7.82     NaN   
1033  110567   山鹰转债  2007-09-05  733567   100  600567  山鹰国际   1.35     NaN   
1034  110026   中海转债  2007-07-02  733026   100  600026  中远海能  14.53     NaN   

         转股价值    债现价  转股溢价率 原股东配售-股权登记日  原股东配售-每股配售额    发行规模   

## 可复用：按债券代码或正股代码自动拉取并估值

`analyze_convertible_bond` 接受 `bond_code` 或 `stock_code`（二选一即可），自动完成：

1. 匹配转债与正股；
2. 拉取正股前复权日线并估算年化波动率；
3. 拉取转债资料、计算剩余期限；
4. 组装参数并调用 `crr_convertible_basic` 估值。

接口不能稳定提供或解析的条款会明确报错，可通过 `coupon=` 和 `P_R=` 覆盖，避免静默使用错误默认值。


In [22]:
# 这段是vibe出来的，实现可以复用地调接口

import re
import warnings


def _six_digit_code(code, field_name="代码"):
    """把 sh123117、SZ300529、123117 等统一成 6 位数字代码。"""
    match = re.search(r"(\d{6})", str(code).strip())
    if not match:
        raise ValueError(f"{field_name}必须包含 6 位数字，收到：{code!r}")
    return match.group(1)


def _sina_symbol(code):
    """将沪深代码转为 AKShare 新浪接口使用的 sh/sz 前缀格式。"""
    code = _six_digit_code(code)
    if code.startswith(("5", "6", "9")):
        return "sh" + code
    if code.startswith(("0", "1", "2", "3")):
        return "sz" + code
    if code.startswith(("4", "8")):
        raise ValueError(f"{code} 看起来是北交所代码，当前使用的新浪日线接口不支持")
    raise ValueError(f"无法判断代码 {code} 所属交易所")


def _profile_value(profile, candidate_items):
    """从 bond_cb_profile_sina 返回的 item/value 表中按候选字段取值。"""
    if not {"item", "value"}.issubset(profile.columns):
        return None
    items = profile["item"].astype(str).str.strip()
    for candidate in candidate_items:
        exact = profile.loc[items.eq(candidate), "value"]
        if not exact.empty and pd.notna(exact.iloc[0]):
            return exact.iloc[0]
    for candidate in candidate_items:
        partial = profile.loc[items.str.contains(candidate, regex=False, na=False), "value"]
        if not partial.empty and pd.notna(partial.iloc[0]):
            return partial.iloc[0]
    return None


def _first_number(value):
    if value is None or pd.isna(value):
        return None
    match = re.search(r"-?\d+(?:\.\d+)?", str(value).replace(",", ""))
    return float(match.group()) if match else None


def _infer_coupon(profile, as_of):
    """尽量从票息说明中选取当前计息年度的票面利率。"""
    raw = _profile_value(profile, ["当前利率", "票面利率", "票面年利率"])
    if raw is None:
        return None
    percentages = [float(x) / 100 for x in re.findall(r"(\d+(?:\.\d+)?)\s*%", str(raw))]
    if not percentages:
        value = _first_number(raw)
        return value / 100 if value is not None and value > 0.1 else value
    if len(percentages) == 1:
        return percentages[0]

    issue_raw = _profile_value(profile, ["起息日", "发行日期", "发行起始日"])
    issue_date = pd.to_datetime(issue_raw, errors="coerce")
    if pd.isna(issue_date):
        return None
    coupon_year = max(1, int(np.ceil((as_of - issue_date.normalize()).days / 365.25)))
    return percentages[min(coupon_year - 1, len(percentages) - 1)]


def _resolve_bond_row(bond_table, bond_code=None, stock_code=None):
    """按债券代码或正股代码定位唯一一行，并防止一股多债时选错。"""
    required = {"债券代码", "债券简称", "正股代码", "转股价"}
    missing = required.difference(bond_table.columns)
    if missing:
        raise KeyError(f"bond_zh_cov 缺少字段：{sorted(missing)}")

    work = bond_table.copy()
    bond_codes = work["债券代码"].astype(str).str.extract(r"(\d{6})", expand=False)
    stock_codes = work["正股代码"].astype(str).str.extract(r"(\d{6})", expand=False)
    mask = pd.Series(True, index=work.index)
    if bond_code is not None:
        mask &= bond_codes.eq(_six_digit_code(bond_code, "债券代码"))
    if stock_code is not None:
        mask &= stock_codes.eq(_six_digit_code(stock_code, "股票代码"))
    matched = work.loc[mask]

    if matched.empty:
        query = f"bond_code={bond_code!r}, stock_code={stock_code!r}"
        raise LookupError(f"没有在 bond_zh_cov 中找到 {query} 对应的转债")
    if len(matched) > 1:
        choices = matched[["债券代码", "债券简称", "正股代码"]].to_dict("records")
        raise ValueError(f"匹配到多只转债，请改用 bond_code 指定：{choices}")
    return matched.iloc[0].copy()


def analyze_convertible_bond(
    bond_code=None,
    stock_code=None,
    *,
    as_of=None,
    lookback_years=1,
    adjust="qfq",
    trading_days=252,
    coupon=None,
    P_R=None,
    r=0.02,
    b=0.0,
    spread=0.01,
    lo=70.0,
    hi=130.0,
    t_conv=0.0,
    t_put=0.0,
    n=200,
    C1=130.0,
    C2=70.0,
    face=100.0,
    P_put=103.0,
    verbose=False,
):
    """拉取一只可转债及其正股数据，并用 CRR 模型计算理论价。

    Parameters
    ----------
    bond_code, stock_code : str, optional
        至少传一个。支持 ``123117``、``sz123117`` 等格式。只传股票代码时，
        若对应多只转债会报错并列出候选项。
    as_of : date-like, optional
        估值日；默认今天。股票历史数据不会使用该日之后的数据。
    lookback_years : int or float
        用于估计历史波动率的回看年数。
    coupon, P_R : float, optional
        年票息率（如 1.8% 写 0.018）与到期赎回价。默认尝试从转债资料解析；
        解析失败时要求显式传入，避免使用不可靠的默认值。

    Returns
    -------
    dict
        ``price``、``params``、``summary``、``bond``、``profile``、
        ``stock_history``。
    """
    if bond_code is None and stock_code is None:
        raise ValueError("bond_code 和 stock_code 至少传一个")
    if lookback_years <= 0:
        raise ValueError("lookback_years 必须大于 0")
    if n <= 0:
        raise ValueError("n 必须为正整数")

    as_of = pd.Timestamp.today().normalize() if as_of is None else pd.Timestamp(as_of).normalize()
    bond_table = bond_zh_cov_df
    row = _resolve_bond_row(bond_table, bond_code=bond_code, stock_code=stock_code)

    resolved_bond_code = _six_digit_code(row["债券代码"], "债券代码")
    resolved_stock_code = _six_digit_code(row["正股代码"], "股票代码")
    bond_symbol = _sina_symbol(resolved_bond_code)
    stock_symbol = _sina_symbol(resolved_stock_code)

    profile = ak.bond_cb_profile_sina(symbol=bond_symbol)
    maturity_raw = _profile_value(profile, ["到期日", "到期日期", "债券到期日"])
    maturity = pd.to_datetime(maturity_raw, errors="coerce")
    if pd.isna(maturity):
        raise ValueError(f"无法从 {resolved_bond_code} 的资料中解析到期日，原始值：{maturity_raw!r}")
    maturity = maturity.normalize()
    T = (maturity - as_of).days / 365.0
    if T <= 0:
        raise ValueError(f"{resolved_bond_code} 在估值日 {as_of.date()} 已到期")

    start_date = as_of - pd.Timedelta(days=round(365.25 * lookback_years))
    stock_history = ak.stock_zh_a_daily(
        symbol=stock_symbol,
        start_date=start_date.strftime("%Y%m%d"),
        end_date=as_of.strftime("%Y%m%d"),
        adjust=adjust,
    )
    close_col = "close" if "close" in stock_history.columns else "收盘"
    if close_col not in stock_history.columns:
        raise KeyError(f"股票日线数据中找不到收盘价字段，现有字段：{list(stock_history.columns)}")
    closes = pd.to_numeric(stock_history[close_col], errors="coerce").dropna()
    closes = closes[closes > 0]
    if len(closes) < 2:
        raise ValueError(f"{resolved_stock_code} 在回看区间内没有足够的有效收盘价")
    log_returns = np.log(closes / closes.shift(1)).dropna()
    sigma = float(log_returns.std(ddof=1) * np.sqrt(trading_days))
    if not np.isfinite(sigma) or sigma <= 0:
        raise ValueError(f"无法得到有效年化波动率：{sigma}")

    K = float(pd.to_numeric(row["转股价"], errors="coerce"))
    if not np.isfinite(K) or K <= 0:
        raise ValueError(f"转股价无效：{row['转股价']!r}")
    market_stock_price = pd.to_numeric(row.get("正股价", np.nan), errors="coerce")
    S = float(market_stock_price) if np.isfinite(market_stock_price) and market_stock_price > 0 else float(closes.iloc[-1])

    if coupon is None:
        coupon = _infer_coupon(profile, as_of)
    if P_R is None:
        redemption_raw = _profile_value(profile, ["到期赎回价", "到期赎回价格", "到期本息和"])
        P_R = _first_number(redemption_raw)
    missing_terms = [name for name, value in {"coupon": coupon, "P_R": P_R}.items() if value is None]
    if missing_terms:
        raise ValueError(
            f"接口资料无法可靠解析 {', '.join(missing_terms)}；请在调用时显式传入，例如 "
            "coupon=0.018, P_R=110"
        )

    params = {
        "S": S, "K": K, "sigma": sigma, "r": float(r), "T": T,
        "b": float(b), "P_R": float(P_R), "coupon": float(coupon),
        "spread": float(spread), "lo": float(lo), "hi": float(hi),
        "t_conv": float(t_conv), "t_put": float(t_put),
    }
    price = crr_convertible_basic(
        **params, n=int(n), C1=float(C1), C2=float(C2), face=float(face),
        P_put=float(P_put), verbose=verbose,
    )
    summary = pd.Series({
        "债券代码": resolved_bond_code,
        "债券简称": row["债券简称"],
        "正股代码": resolved_stock_code,
        "估值日": as_of.date(),
        "到期日": maturity.date(),
        "剩余年限": T,
        "正股价": S,
        "转股价": K,
        "年化波动率": sigma,
        "票息率": float(coupon),
        "到期赎回价": float(P_R),
        "理论价": price,
    })
    return {
        "price": price,
        "params": params,
        "summary": summary,
        "bond": row,
        "profile": profile,
        "stock_history": stock_history,
    }


In [31]:
bond_cb_profile_sina_df = ak.bond_cb_profile_sina(symbol="sz123257")
print(bond_cb_profile_sina_df)

        item                                             value
0       债券名称                  2025年安克创新科技股份有限公司向不特定对象发行可转换公司债券
1       债券简称                                              安克转债
2       债券代码                                          sz123257
3       债券类型                                            可转换企业债
4    债券面值（元）                                               100
5    债券年限（年）                                                 6
6    票面利率（%）                                                --
7        到期日                                        2031-06-16
8        兑付日                                        2031-06-16
9        摘牌日                                                --
10      计息方式                                              递进利率
11      利率说明  第一年0.2%、第二年0.4%、第三年0.6%、第四年1.5%、第五年1.8%、第六年2.0%。
12      付息方式                                             周期性付息
13      起息日期                                        2025-06-16
14      止息日期                                        203

In [32]:
# 示例 1：输入债券代码（前缀可省略）
# 健帆转债的票息和到期赎回价沿用原 notebook 的人工核对值。
result = analyze_convertible_bond(
    bond_code="123257",
    coupon=0.004,
    P_R=108,
    r=0.02,
    spread=0.01,
    n=200,
)
display(result["summary"])

# 为兼容后续单元格，保留原变量名
params_cb = result["params"]

# 示例 2：也可以只输入股票代码；若一股对应多债，函数会列出候选债券代码
# result = analyze_convertible_bond(stock_code="300529", coupon=0.018, P_R=110)

print(params_cb)


,0
债券代码,123257
债券简称,安克转债
正股代码,300866
估值日,2026-07-20
到期日,2031-06-16
剩余年限,4.909589
正股价,115.71
转股价,108.86
年化波动率,0.45148
票息率,0.004


{'S': 115.71, 'K': 108.86, 'sigma': 0.4514804604668258, 'r': 0.02, 'T': 4.909589041095891, 'b': 0.0, 'P_R': 108.0, 'coupon': 0.004, 'spread': 0.01, 'lo': 70.0, 'hi': 130.0, 't_conv': 0.0, 't_put': 0.0}


### 为什么少8块？
- 下修条款没做
- 模型精度？可以玩一下TF
- 其他误差？

但这也差太多了

### 关键结果

- 模型理论价 $\approx 108$ 元，市场债现价 $115.7$ 元，**低约 8 元**。
- 这只券深度价外，理论价几乎等于纯债底（到期赎回价折现 + 票息）。市场价高出的部分主要来自**下修条款预期**：市场押注发行人下调转股价 $K$，让转股期权重新有价值。暂时没做下修，系统性低估。
- 敏感度分析佐证：调参后最多到 $109$，补不平这 $8$ 元，说明缺的是结构性期权而非参数没调好。
- 附带观察：对这只深度价外券，树的层数 $n$ 从 $5$ 到 $1000$ 价格几乎不变，因为它本质是一张债、期权成分极小；平价 $100$ 附近的券才需要大 $n$

### 下一步
1. 加下修条款

2. 已封装自动拉取函数；下一步可选一张平价接近 100 的券重新测试

3. 实现 TF 模型：价值拆成 $B+E$ 两条数组分别倒推，比"平价插值折现"更规范地处理信用风险。


In [33]:
theoretical_price = result["price"]
market_price = float(result["bond"]["债现价"])

difference = theoretical_price - market_price
difference_pct = difference / market_price

print(f"模型理论价：{theoretical_price:.2f}")
print(f"市场债现价：{market_price:.2f}")
print(f"理论价 - 市场价：{difference:.2f}")
print(f"相对差异：{difference_pct:.2%}")


模型理论价：121.28
市场债现价：156.17
理论价 - 市场价：-34.89
相对差异：-22.34%
